## Text chunked and the check the quality of chunk

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

True

## Setup configuration

In [2]:
class Config:
    # setup mistral configuration
    mistral_api_key = os.getenv("MISTRAL_API_KEY")
    mistral_chat_model = os.getenv("MISTRAL_CHAT_MODEL")
    mistral_embed_model = "mistral-embed"
    mistral_embed_dimension =  int(os.getenv("MISTRAL_EMBED_DIMENSION")) or 1024

## Setup LLM Service

In [3]:
from langchain_mistralai import ChatMistralAI, MistralAIEmbeddings

class MistralService:
    chatModel : ChatMistralAI
    embeddingModel : MistralAIEmbeddings

    def __init__(self):
        self.chatModel = self.connectMistralChatModel()
        self.embeddingModel = self.connectMistralEmbedModel()

    def connectMistralChatModel(self, model_name : str = Config.mistral_chat_model) -> ChatMistralAI :
        try:
            return ChatMistralAI(
                api_key = Config.mistral_api_key,
                model = model_name
            )
        except Exception as e:
            print(f"Something went wrong in the {model_name} connection...")
            raise
        
    def connectMistralEmbedModel(self, model_name: str = Config.mistral_embed_model) -> MistralAIEmbeddings :
        try:
            return MistralAIEmbeddings(
                api_key = Config.mistral_api_key,
                model = model_name
            )
        except Exception as e:
            print(f"Something went wrong in the {model_name} connection...")
            raise
    
    def getChatModel(self) -> ChatMistralAI:
        return self.chatModel
    
    def getEmbedModel(self) -> MistralAIEmbeddings:
        return self.embeddingModel

## Setup mock document data

In [5]:
from langchain_core.documents import Document

document_list = [Document(metadata={'document_name': 'ai.pdf', 'document_created_at': '2026-03-13T22:04:58.558621', 'text': 'At its simplest, Artificial Intelligence (AI)  is the quest to build machines that can \nperform tasks usually requiring human intelligence.\nRather than just following a rigid "if this, then that" script, AI uses data to learn patterns,  \nmake decisions, and solve problems.\nHow It Actually Works\nThink of standard software like a calculator: it follows exact rules to get an exact result. \nAI is more like a digital apprentice : you show it millions of examples, and it learns how \nto mimic the desired output.\n•Machine Learning (ML):  The engine of AI. It’s the process of using algorithms to parse \ndata, learn from it, and then make a determination or prediction.\n•Neural Networks:  A type of ML inspired by the human brain’s structure, using layers of \n"nodes" to process complex information.\n•Generative AI: The current "superstar" (like me!). These models don\'t just analyze data; \nthey use what they’ve learned to create brand-new content like text, images, or music.\nWhere You See It Today'}, page_content='At its simplest, Artificial Intelligence (AI)  is the quest to build machines that can \nperform tasks usually requiring human intelligence.\nRather than just following a rigid "if this, then that" script, AI uses data to learn patterns,  \nmake decisions, and solve problems.\nHow It Actually Works\nThink of standard software like a calculator: it follows exact rules to get an exact result. \nAI is more like a digital apprentice : you show it millions of examples, and it learns how \nto mimic the desired output.\n•Machine Learning (ML):  The engine of AI. It’s the process of using algorithms to parse \ndata, learn from it, and then make a determination or prediction.\n•Neural Networks:  A type of ML inspired by the human brain’s structure, using layers of \n"nodes" to process complex information.\n•Generative AI: The current "superstar" (like me!). These models don\'t just analyze data; \nthey use what they’ve learned to create brand-new content like text, images, or music.\nWhere You See It Today'), Document(metadata={'document_name': 'ai.pdf', 'document_created_at': '2026-03-13T22:04:58.558621', 'text': 'they use what they’ve learned to create brand-new content like text, images, or music.\nWhere You See It Today\n•Personalization:  Netflix suggesting your next binge-watch or Spotify’s "Discover \nWeekly."\n•Natural Language:  Virtual assistants (Siri, Alexa) and real-time translation tools.\n•Vision: FaceID on your phone or cars that can detect pedestrians.\n•Efficiency: Detecting credit card fraud in milliseconds or optimizing delivery routes for \nmail.\nWhy It Matters\nAI isn\'t about building "Terminators." It’s about augmentation. It handles the heavy \nlifting of data processing so humans can focus on strategy, creativity, and empathy. \nWhile it\'s incredibly powerful, it\'s still just a tool—it lacks true consciousness and relies \nentirely on the quality of the data we give it.\n•Neural networks  explained\n•Ethical risks like bias\n•AI vs. Automation\n•Future job impact\nWhat\'s your biggest curiosity about this tech?'}, page_content='they use what they’ve learned to create brand-new content like text, images, or music.\nWhere You See It Today\n•Personalization:  Netflix suggesting your next binge-watch or Spotify’s "Discover \nWeekly."\n•Natural Language:  Virtual assistants (Siri, Alexa) and real-time translation tools.\n•Vision: FaceID on your phone or cars that can detect pedestrians.\n•Efficiency: Detecting credit card fraud in milliseconds or optimizing delivery routes for \nmail.\nWhy It Matters\nAI isn\'t about building "Terminators." It’s about augmentation. It handles the heavy \nlifting of data processing so humans can focus on strategy, creativity, and empathy. \nWhile it\'s incredibly powerful, it\'s still just a tool—it lacks true consciousness and relies \nentirely on the quality of the data we give it.\n•Neural networks  explained\n•Ethical risks like bias\n•AI vs. Automation\n•Future job impact\nWhat\'s your biggest curiosity about this tech?')]

## Document chunk validity checker

In [6]:
from langchain_core.prompts import PromptTemplate

prompt_message = """
    you need to act as content quality checker. Your task is the generate the value how much that statement quality is ? is that wanted valud content or its just normal statement

    statement:
    {statement}
"""

prompt_template = PromptTemplate.from_template(prompt_message)

def pareprePrompt(content: str) -> PromptTemplate:
    return prompt_template.invoke({
        "statement" : content
    })

In [8]:
from pydantic import BaseModel, Field

class ChunkQualityOutput(BaseModel):
    quality_score : float = Field(..., ge= 0, le=1 )

In [11]:
mistral = MistralService()
llm = mistral.getChatModel()
structured_llm = llm.with_structured_output(ChunkQualityOutput)

/home/buddhika-madusanka/projects/industrial-rag-system/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [24]:
def getChunkQuality(doc: Document):
    prompt = pareprePrompt(doc.page_content)
    chunk_quality_repsponse = structured_llm.invoke(prompt)

    doc.metadata['chunk_quality'] = chunk_quality_repsponse.quality_score
    return chunk_quality_repsponse.quality_score

def qualifiedQualityChunks(docs: list[Document]):
    qualifiedChunks = []
    for doc in docs:
        quality_score = getChunkQuality(doc)

        if quality_score > 0.6:
            qualifiedChunks.append(doc)
    
    for i, doc in enumerate(docs, start= 1):
        doc.metadata['chunk_arranged_number'] = i

    return qualifiedChunks

qualifiedQualityChunks(document_list)

[Document(metadata={'document_name': 'ai.pdf', 'document_created_at': '2026-03-13T22:04:58.558621', 'text': 'At its simplest, Artificial Intelligence (AI)  is the quest to build machines that can \nperform tasks usually requiring human intelligence.\nRather than just following a rigid "if this, then that" script, AI uses data to learn patterns,  \nmake decisions, and solve problems.\nHow It Actually Works\nThink of standard software like a calculator: it follows exact rules to get an exact result. \nAI is more like a digital apprentice : you show it millions of examples, and it learns how \nto mimic the desired output.\n•Machine Learning (ML):  The engine of AI. It’s the process of using algorithms to parse \ndata, learn from it, and then make a determination or prediction.\n•Neural Networks:  A type of ML inspired by the human brain’s structure, using layers of \n"nodes" to process complex information.\n•Generative AI: The current "superstar" (like me!). These models don\'t just analy

In [ ]:
[Document(metadata={'document_name': 'ai.pdf', 'document_created_at': '2026-03-13T22:04:58.558621', 'text': 'At its simplest, Artificial Intelligence (AI)  is the quest to build machines that can \nperform tasks usually requiring human intelligence.\nRather than just following a rigid "if this, then that" script, AI uses data to learn patterns,  \nmake decisions, and solve problems.\nHow It Actually Works\nThink of standard software like a calculator: it follows exact rules to get an exact result. \nAI is more like a digital apprentice : you show it millions of examples, and it learns how \nto mimic the desired output.\n•Machine Learning (ML):  The engine of AI. It’s the process of using algorithms to parse \ndata, learn from it, and then make a determination or prediction.\n•Neural Networks:  A type of ML inspired by the human brain’s structure, using layers of \n"nodes" to process complex information.\n•Generative AI: The current "superstar" (like me!). These models don\'t just analyze data; \nthey use what they’ve learned to create brand-new content like text, images, or music.\nWhere You See It Today', 'chunk_arrange_number': 1, 'chunk_quality': 0.95, 'chunk_number': 1, 'chunk_arranged_number': 1}, page_content='At its simplest, Artificial Intelligence (AI)  is the quest to build machines that can \nperform tasks usually requiring human intelligence.\nRather than just following a rigid "if this, then that" script, AI uses data to learn patterns,  \nmake decisions, and solve problems.\nHow It Actually Works\nThink of standard software like a calculator: it follows exact rules to get an exact result. \nAI is more like a digital apprentice : you show it millions of examples, and it learns how \nto mimic the desired output.\n•Machine Learning (ML):  The engine of AI. It’s the process of using algorithms to parse \ndata, learn from it, and then make a determination or prediction.\n•Neural Networks:  A type of ML inspired by the human brain’s structure, using layers of \n"nodes" to process complex information.\n•Generative AI: The current "superstar" (like me!). These models don\'t just analyze data; \nthey use what they’ve learned to create brand-new content like text, images, or music.\nWhere You See It Today'),
 Document(metadata={'document_name': 'ai.pdf', 'document_created_at': '2026-03-13T22:04:58.558621', 'text': 'they use what they’ve learned to create brand-new content like text, images, or music.\nWhere You See It Today\n•Personalization:  Netflix suggesting your next binge-watch or Spotify’s "Discover \nWeekly."\n•Natural Language:  Virtual assistants (Siri, Alexa) and real-time translation tools.\n•Vision: FaceID on your phone or cars that can detect pedestrians.\n•Efficiency: Detecting credit card fraud in milliseconds or optimizing delivery routes for \nmail.\nWhy It Matters\nAI isn\'t about building "Terminators." It’s about augmentation. It handles the heavy \nlifting of data processing so humans can focus on strategy, creativity, and empathy. \nWhile it\'s incredibly powerful, it\'s still just a tool—it lacks true consciousness and relies \nentirely on the quality of the data we give it.\n•Neural networks  explained\n•Ethical risks like bias\n•AI vs. Automation\n•Future job impact\nWhat\'s your biggest curiosity about this tech?', 'chunk_arrange_number': 2, 'chunk_quality': 0.95, 'chunk_number': 2, 'chunk_arranged_number': 2}, page_content='they use what they’ve learned to create brand-new content like text, images, or music.\nWhere You See It Today\n•Personalization:  Netflix suggesting your next binge-watch or Spotify’s "Discover \nWeekly."\n•Natural Language:  Virtual assistants (Siri, Alexa) and real-time translation tools.\n•Vision: FaceID on your phone or cars that can detect pedestrians.\n•Efficiency: Detecting credit card fraud in milliseconds or optimizing delivery routes for \nmail.\nWhy It Matters\nAI isn\'t about building "Terminators." It’s about augmentation. It handles the heavy \nlifting of data processing so humans can focus on strategy, creativity, and empathy. \nWhile it\'s incredibly powerful, it\'s still just a tool—it lacks true consciousness and relies \nentirely on the quality of the data we give it.\n•Neural networks  explained\n•Ethical risks like bias\n•AI vs. Automation\n•Future job impact\nWhat\'s your biggest curiosity about this tech?')]